# Episodic MoBA PPO (Colab launcher)

Thin launcher only: implementation/training logic stays in the repository. Run cells top-to-bottom; a failed baseline gate aborts the notebook.

In [ ]:
from pathlib import Path
import os, subprocess
# Point these at the repository containing this retrofit, not the upstream base.
REPO_URL="REPLACE_WITH_IMPLEMENTATION_REPO_URL"
REPO_COMMIT="REPLACE_WITH_IMPLEMENTATION_COMMIT"
DRIVE_ROOT="/content/drive/MyDrive/episodic-moba-ppo"
WANDB_ENTITY="REPLACE_ME"
arm="trxl_moba"; seed=1
REPO_DIR=Path("/content/episodic-moba-ppo")
BASE_CONFIG="configs/trxl_command40.yaml" if arm=="trxl" else "configs/trxl_moba_command40.yaml"
RUN_ID=f"{arm}-command40-seed{seed}"
RESOLVED_CONFIG=Path("/content")/f"{RUN_ID}.yaml"
def run(*args,cwd=REPO_DIR,env=None):
    e=os.environ.copy(); e.update({k:str(v) for k,v in (env or {}).items()})
    print("$"," ".join(map(str,args))); return subprocess.run([str(a) for a in args],cwd=cwd,env=e,check=True)


In [ ]:
if REPO_URL.startswith("REPLACE_") or REPO_COMMIT.startswith("REPLACE_"):
    raise ValueError("Set REPO_URL and REPO_COMMIT to the published implementation revision")
if not REPO_DIR.exists(): run("git","clone",REPO_URL,REPO_DIR,cwd=Path("/content"))
run("git","fetch","--tags","--force")
run("git","checkout","--detach",REPO_COMMIT)
actual=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
assert actual == REPO_COMMIT, (actual, REPO_COMMIT)


In [ ]:
from google.colab import auth, drive
drive.mount("/content/drive"); auth.authenticate_user()
import wandb; wandb.login()
Path(DRIVE_ROOT).mkdir(parents=True,exist_ok=True)
run("python","-m","pip","install","-q","uv")
run("uv","python","install","3.11")
run("uv","sync","--frozen","--python","3.11")


## Mandatory baseline gate

The command exits non-zero if either threshold fails; `run` propagates that failure and prevents later cells from running.

In [ ]:
run("uv","run","eval-pretrained","--config","configs/pretrained_eval.yaml","--repo-root",".","--output","results/baseline_reference.json")


## Resolve, train, resume, evaluate, analyze

Create a run-specific YAML without mutating checked-in configs. Resume only from a durable update directory containing `commit_success.json`.


In [ ]:
import yaml
with (REPO_DIR/BASE_CONFIG).open() as f: cfg=yaml.safe_load(f)
cfg["seeds"]["model"]=seed
cfg["wandb"]["entity"]=WANDB_ENTITY; cfg["wandb"]["run_name"]=RUN_ID
cfg["drive"]["root"]=DRIVE_ROOT
cfg["checkpointing"]["local_dir"]=f"/content/checkpoints/{RUN_ID}"
cfg["checkpointing"]["drive_dir"]=f"checkpoints/{RUN_ID}"
with RESOLVED_CONFIG.open("w") as f: yaml.safe_dump(cfg,f,sort_keys=False)
run("uv","run","train","--config",RESOLVED_CONFIG,"--repo-root",".",env={"PYTHONHASHSEED":seed})


In [ ]:
# Use update-00062 only after the shared extension gate authorizes both arms.
CHECKPOINT_DIR=Path(DRIVE_ROOT)/"checkpoints"/RUN_ID/"update-00031"
EVALUATION_OUTPUT=REPO_DIR/"results"/f"{RUN_ID}.json"
run("uv","run","evaluate","--checkpoint",CHECKPOINT_DIR,"--arm",arm,"--model-seed",seed,"--output",EVALUATION_OUTPUT)
if arm == "trxl_moba":
    run("uv","run","analyze-retrieval","--config","configs/analyze_retrieval.yaml")
